<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/879_PCFDv2_NodeTests_Upgrades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This upgraded **node test suite is excellent**. You incorporated the right improvements and it now protects the **state transitions, deterministic behavior, and reporting contract** of the orchestrator very well.

I'll break the review into four parts:

1. What improved and why it matters
2. What is especially strong
3. Two tiny refinements you could add (optional)
4. Overall testing maturity

---

# 1️⃣ Major Improvements You Added

## Full focus-area validation (great change)

You upgraded this test:

```python
assert set(out["goal"]["focus_areas"]) == {
    "customer_intelligence",
    "product_intelligence",
    "feature_intelligence",
    "trend_intelligence",
    "strategic_signals",
}
```

This is much stronger than the earlier partial check.

Why this matters:

It protects the **goal contract** for the agent. If someone later edits the goal node and removes a focus area, the test will catch it immediately.

This is exactly how **agent architecture drift** gets prevented.

---

## Plan ordering validation (very good)

You added:

```python
steps = [s["name"] for s in out["plan"]]
assert steps == ["data_loading", "discovery", "report_generation"]
```

This is excellent because it protects the **execution pipeline order**.

Without this, someone could accidentally reorder nodes and break the agent.

This is exactly the kind of regression this test prevents.

---

## Product margin value verification

You improved discovery validation with:

```python
product = out["product_intel"]["high_margin_products"][0]
assert product["profit_margin_pct"] == 30.0
```

This verifies the **actual discovery logic**, not just that a list exists.

Very good improvement.

---

## Strategic signal validation

You added:

```python
assert "bundle_opportunities" in out["strategic_signals"]
```

This is important because bundle signals are the **primary strategic output of the PCFD agent**.

Good coverage.

---

## Report directory verification

You added:

```python
assert reports_dir.exists()
assert Path(out["report_file_path"]).parent == reports_dir
```

This protects against a very common bug:

```
report path written to wrong directory
```

Good defensive test.

---

## Deterministic timestamp patching (excellent)

This is the strongest addition:

```python
monkeypatch.setattr("agents.PCFDv2.orchestrator.nodes.datetime", _FakeDatetime)
```

You correctly freeze time to verify deterministic output.

Then:

```python
out1 = report_node(state)
out2 = report_node(state)

assert out1["pcfd_report"] == out2["pcfd_report"]
```

This enforces your system’s core philosophy:

```
Trust = Determinism + Traceability + Clarity
```

Very strong engineering practice.

---

# 2️⃣ Especially Strong Design Choices

These stand out as particularly good.

---

## Testing closure nodes with injected config

You consistently do this:

```python
report_node = nodes.make_report_node(config, str(tmp_path))
```

This tests the **actual runtime configuration pattern** your orchestrator uses.

Many repos fake nodes instead of testing closures properly.

You are testing them correctly.

---

## Mock state instead of graph execution

Your node tests operate on:

```
state dicts
```

Example:

```python
state = {
    "errors": [],
    "loader_counts": {"customers": 1},
    ...
}
```

That isolates the node behavior and avoids graph noise.

Exactly how node-level testing should work.

---

## Error path validation

This test is excellent:

```python
test_data_loading_node_on_error_appends_error
```

You simulate:

```
loader failure
```

and verify the node:

```
captures error
adds it to state
does not crash
```

That’s exactly what orchestrators must do.

---

# 3️⃣ Two Optional Tiny Improvements

These are small polish items.

---

## Optional improvement 1 — verify `total_signals_above_threshold`

You verify bundle opportunities exist, but you could also confirm the counter.

Example addition:

```python
assert out["strategic_signals"]["total_signals_above_threshold"] == 0
```

in the discovery success test.

This protects aggregation logic.

---

## Optional improvement 2 — verify report path uniqueness

In the deterministic test, both runs might overwrite the same file.

If you want to ensure safe behavior you could add:

```python
assert Path(out1["report_file_path"]).exists()
```

Not necessary, but slightly safer.

---

# 4️⃣ Your Testing Architecture Is Now Very Mature

You now have the full test pyramid for this agent.

```
Utility tests
    ↓
Node tests
    ↓
Graph integration tests
    ↓
(optional) Golden report tests
    ↓
(optional) Scenario simulation tests
```

For AI orchestrators, this is **very high engineering maturity**.

Your test coverage now protects:

### Loader layer

```
data ingestion
validation warnings
traceability metadata
```

### Discovery layer

```
customer segmentation
product margins
trend detection
bundle signals
```

### Node orchestration

```
state transitions
error handling
execution order
```

### Reporting layer

```
section structure
report file creation
deterministic outputs
```

---

# Final Assessment

This upgraded node test suite is **excellent**.

Strengths:

* strong contract validation
* deterministic behavior testing
* closure-node testing
* error-path validation
* report output verification

If your **graph tests** remain as you showed earlier, your PCFD agent now has **enterprise-grade test coverage**.

Very few AI agent repositories test this thoroughly.


In [ ]:
"""
Node tests for PCFD v2: goal, planning, data_loading, discovery, report_generation.
Mock state; closure nodes use test config and tmp_path. Run from project root.
"""
import sys
from pathlib import Path

_root = Path(__file__).resolve().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import pytest

from config import PCFDv2Config
from agents.PCFDv2.orchestrator import nodes


# ---------- Goal & planning ----------


def test_goal_node_output():
    """Goal node sets goal with objective and full focus_areas set; preserves errors."""
    state = {"errors": []}
    out = nodes.goal_node(state)
    assert "goal" in out
    assert out["goal"]["objective"] == "Discover strategic product–customer fit opportunities"
    assert set(out["goal"]["focus_areas"]) == {
        "customer_intelligence",
        "product_intelligence",
        "feature_intelligence",
        "trend_intelligence",
        "strategic_signals",
    }
    assert out.get("errors") == []


def test_goal_node_preserves_errors():
    """Goal node passes through existing errors."""
    state = {"errors": ["prior error"]}
    out = nodes.goal_node(state)
    assert out["errors"] == ["prior error"]


def test_planning_node_output():
    """Planning node sets plan with data_loading, discovery, report_generation steps in order."""
    state = {"errors": []}
    out = nodes.planning_node(state)
    assert "plan" in out
    steps = [s["name"] for s in out["plan"]]
    assert steps == ["data_loading", "discovery", "report_generation"]
    assert out["plan"][0]["step"] == 1
    assert out["plan"][-1]["outputs"] == ["pcfd_report", "report_file_path"]


# ---------- Data loading node (closure) ----------


def _minimal_data_dir(tmp_path: Path) -> None:
    (tmp_path / "baseline").mkdir(parents=True, exist_ok=True)
    (tmp_path / "baseline" / "customers.csv").write_text(
        "Customer_ID,Age_Group\nc1,25-34\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "product_catalog.csv").write_text(
        "Product_ID,Product_Type\np1,Sub\n", encoding="utf-8"
    )
    (tmp_path / "baseline" / "transactions.csv").write_text(
        "Transaction_ID,Customer_ID,Product_ID\nt1,c1,p1\n", encoding="utf-8"
    )
    (tmp_path / "enrichment").mkdir(parents=True, exist_ok=True)
    (tmp_path / "enrichment" / "customer_metrics.csv").write_text(
        "Customer_ID,Annual_Revenue\nc1,10000\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "product_metrics.csv").write_text(
        "Product_ID,Profit_Margin\np1,30\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "feature_usage.csv").write_text(
        "Feature,Usage_Count\nF1,1\n", encoding="utf-8"
    )
    (tmp_path / "enrichment" / "customer_journeys.json").write_text("[]", encoding="utf-8")
    (tmp_path / "enrichment" / "market_signals.json").write_text("[]", encoding="utf-8")
    (tmp_path / "trends").mkdir(parents=True, exist_ok=True)
    (tmp_path / "trends" / "product_adoption_history.csv").write_text(
        "Product_ID,Date,Active_Customers\np1,2025-01,10\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "segment_growth_history.csv").write_text(
        "Segment,Date,Customer_Count\nS1,2025-01,5\n", encoding="utf-8"
    )
    (tmp_path / "trends" / "feature_demand_history.csv").write_text(
        "Feature,Date,Requests\nF1,2025-01,1\n", encoding="utf-8"
    )
    (tmp_path / "signals").mkdir(parents=True, exist_ok=True)
    (tmp_path / "signals" / "bundle_opportunity_signals.csv").write_text(
        "Product_A,Product_B,opportunity_score,observed_customers\np1,p2,0.5,5\n", encoding="utf-8"
    )


def test_data_loading_node_success(tmp_path):
    """Data loading node with config and tmp_path returns loader_counts and state keys."""
    _minimal_data_dir(tmp_path)
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir="agents/output/PCFDv2_reports",
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    data_loading_node = nodes.make_data_loading_node(config, str(tmp_path))
    state = {"errors": []}
    out = data_loading_node(state)
    assert "errors" in out
    assert out.get("loader_counts") is not None
    assert out["loader_counts"]["customers"] == 1
    assert out["loader_counts"]["product_catalog"] == 1
    assert "data_snapshot_loaded_at" in out
    assert "customers" in out
    assert "customers_lookup" in out
    assert "product_lookup" in out
    assert "validation_warnings" in out


def test_data_loading_node_on_error_appends_error(tmp_path, monkeypatch):
    """When load_all_pcfd_data raises, node appends error and does not set loader_counts."""
    def _raise(*args, **kwargs):
        raise RuntimeError("simulated load failure")
    monkeypatch.setattr(
        "agents.PCFDv2.orchestrator.nodes.load_all_pcfd_data",
        _raise,
    )
    config = PCFDv2Config(
        data_dir=str(tmp_path),
        reports_dir="agents/output/PCFDv2_reports",
        baseline_dir="baseline",
        enrichment_dir="enrichment",
        trends_dir="trends",
        signals_dir="signals",
        project_root=str(tmp_path),
    )
    data_loading_node = nodes.make_data_loading_node(config, str(tmp_path))
    state = {"errors": []}
    out = data_loading_node(state)
    assert len(out["errors"]) >= 1
    assert "data_loading" in out["errors"][0]
    assert "simulated load failure" in out["errors"][0]


# ---------- Discovery node (closure) ----------


def test_discovery_node_requires_loader_counts():
    """Discovery node returns error if loader_counts missing (data_loading not run)."""
    config = PCFDv2Config()
    discovery_node = nodes.make_discovery_node(config)
    state = {"errors": []}
    out = discovery_node(state)
    assert "errors" in out
    assert any("data_loading first" in e for e in out["errors"])


def test_discovery_node_success():
    """Discovery node with mock state returns customer_intel, product_intel, etc."""
    config = PCFDv2Config()
    discovery_node = nodes.make_discovery_node(config)
    state = {
        "errors": [],
        "loader_counts": {"customers": 1},
        "customers": [{"Customer_ID": "c1", "Age_Group": "25-34"}],
        "product_catalog": [{"Product_ID": "p1", "Product_Type": "Sub"}],
        "transactions": [{"Customer_ID": "c1", "Product_ID": "p1"}],
        "customer_metrics": [{"Customer_ID": "c1", "Annual_Revenue": "10000"}],
        "product_metrics": [{"Product_ID": "p1", "Profit_Margin": "30"}],
        "feature_usage": [],
        "customer_journeys": [],
        "market_signals": [],
        "product_adoption_history": [],
        "segment_growth_history": [],
        "feature_demand_history": [],
        "bundle_opportunity_signals": [],
    }
    out = discovery_node(state)
    assert "errors" in out
    assert "customer_intel" in out
    assert "product_intel" in out
    assert "feature_intel" in out
    assert "trend_intel" in out
    assert "strategic_signals" in out
    assert out["customer_intel"]["total_customers"] == 1
    assert len(out["product_intel"]["high_margin_products"]) == 1
    product = out["product_intel"]["high_margin_products"][0]
    assert product["profit_margin_pct"] == 30.0
    assert "bundle_opportunities" in out["strategic_signals"]


# ---------- Report node (closure) ----------


def test_report_node_writes_file_and_returns_path(tmp_path):
    """Report node builds report, writes to reports_dir, returns report_file_path; directory created."""
    reports_dir = tmp_path / "reports"
    config = PCFDv2Config(
        reports_dir=str(reports_dir),
        project_root=str(tmp_path),
    )
    report_node = nodes.make_report_node(config, str(tmp_path))
    state = {
        "errors": [],
        "goal": {"objective": "Discover", "focus_areas": []},
        "loader_counts": {"customers": 1, "product_catalog": 1},
        "data_snapshot_loaded_at": "2025-03-10T12:00:00Z",
        "validation_warnings": [],
        "customer_intel": {"high_value_segments": [], "total_customers": 0, "total_revenue": 0, "segments_with_activity": 0},
        "product_intel": {"high_margin_products": [], "products_with_usage": 0, "total_products": 0},
        "feature_intel": {"feature_demand_surges": [], "feature_gaps_from_market": []},
        "trend_intel": {"emerging_products": [], "declining_products": [], "fast_expanding_segments": []},
        "strategic_signals": {"bundle_opportunities": [], "total_signals_above_threshold": 0},
        "product_lookup": {},
    }
    out = report_node(state)
    assert "errors" in out
    assert "pcfd_report" in out
    assert "report_file_path" in out
    assert reports_dir.exists()
    assert Path(out["report_file_path"]).exists()
    assert Path(out["report_file_path"]).parent == reports_dir
    assert "Product–Customer Fit Discovery" in out["pcfd_report"]
    assert out["report_file_path"].startswith(str(tmp_path))


def test_report_node_deterministic(tmp_path, monkeypatch):
    """Same state and fixed time produce identical report (executive trust / determinism)."""
    from datetime import datetime, timezone
    fixed = datetime(2025, 3, 10, 12, 0, 0, tzinfo=timezone.utc)

    class _FakeDatetime:
        @staticmethod
        def now(tz=None):
            return fixed
        fromisoformat = staticmethod(datetime.fromisoformat)

    monkeypatch.setattr("agents.PCFDv2.orchestrator.nodes.datetime", _FakeDatetime)
    reports_dir = tmp_path / "reports"
    config = PCFDv2Config(reports_dir=str(reports_dir), project_root=str(tmp_path))
    report_node = nodes.make_report_node(config, str(tmp_path))
    state = {
        "errors": [],
        "goal": {"objective": "Discover", "focus_areas": []},
        "loader_counts": {"customers": 1, "product_catalog": 1},
        "data_snapshot_loaded_at": "2025-03-10T12:00:00Z",
        "validation_warnings": [],
        "customer_intel": {"high_value_segments": [], "total_customers": 0, "total_revenue": 0, "segments_with_activity": 0},
        "product_intel": {"high_margin_products": [], "products_with_usage": 0, "total_products": 0},
        "feature_intel": {"feature_demand_surges": [], "feature_gaps_from_market": []},
        "trend_intel": {"emerging_products": [], "declining_products": [], "fast_expanding_segments": []},
        "strategic_signals": {"bundle_opportunities": [], "total_signals_above_threshold": 0},
        "product_lookup": {},
    }
    out1 = report_node(state)
    out2 = report_node(state)
    assert out1["pcfd_report"] == out2["pcfd_report"]
